# Notebook 4 — CMF Unlearning: cmf_static vs cmf_dynamic (3:7 Cross-Class Split)

**Experiment:** ReGUn benchmark — Group B, two CMF strategies side-by-side.

**Prerequisite:** Run **Notebook 1** first and attach its output as a Kaggle dataset.
Set `CKPT_DATASET_DIR` below.

**Two CMF algorithms** selected via `classifier_strategy`:

| Strategy | Key behaviour |
|----------|---------------|
| `cmf_static` | Original paper: W rebuilt via closed-form CMF each epoch, then **frozen** during encoder update |
| `cmf_dynamic` | AlternatingCMF: W warm-started via CMF reconstruction, then updated as a **real trainable parameter** in alternating phases with θ |

Both run on the same fixed 3:7 cross-class split from Notebook 1 (seeds 0,1,2).

| Stage | Description |
|-------|-------------|
| **A** | Environment setup, load config |
| **B** | Load splits, dataset, Θ_o |
| **C** | CMF algorithm implementations |
| **D** | Experiment matrix runner |
| **E** | Results CSV + pivoted cmf_static vs cmf_dynamic table |
| **F** | Written summary (a)–(d) |

## A. Environment Setup

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, copy, random, argparse, collections, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SET THIS to the Kaggle dataset mount path from Notebook 1.
# ══════════════════════════════════════════════════════════════════════
CKPT_DATASET_DIR = '/kaggle/input/regun-notebook1'  # ← EDIT THIS

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/regun_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/regun/regun_config.json',
    f'{CKPT_DATASET_DIR}/regun/regun_config.json',
]
config_path = None
CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p
        CKPT_ROOT_NB1 = os.path.dirname(_p)
        break
assert config_path is not None, 'regun_config.json not found. Check CKPT_DATASET_DIR.'

with open(config_path) as f:
    CFG = json.load(f)

TEST_MODE         = CFG['TEST_MODE']
TEST_FRACTION     = CFG['TEST_FRACTION']
_MODE_TAG         = CFG['_MODE_TAG']
DATASET           = CFG['DATASET']
ARCH              = CFG['ARCH']
IS_VIT            = CFG['IS_VIT']
NUM_CLASSES       = CFG['NUM_CLASSES']
CLASS_LABEL_NAMES = CFG['CLASS_LABEL_NAMES']
SPLIT_SEEDS       = CFG['SPLIT_SEEDS']
FORGET_FRACTION   = CFG['FORGET_FRACTION']
PRETRAIN_LR       = CFG['PRETRAIN_LR']
PRETRAIN_EPOCHS   = CFG['PRETRAIN_EPOCHS']
PRETRAIN_BS       = CFG['PRETRAIN_BS']
PRETRAIN_PATIENCE = CFG['PRETRAIN_PATIENCE']
_TOTAL     = {DATASET: CFG['TOTAL']}
_PER_CLASS = {DATASET: CFG['PER_CLASS']}

_old_root = CFG['CKPT_ROOT']
def _repath(p): return p.replace(_old_root, CKPT_ROOT_NB1)
CKPT_PRETRAIN = _repath(CFG['CKPT_PRETRAIN'])
SPLIT_DIR     = _repath(CFG['SPLIT_DIR'])

for p, name in [(CKPT_PRETRAIN, 'pre_train')]:
    print(f'  [{"OK" if os.path.exists(p) else "MISSING"}] {name}: {p}')

DATA_PATH     = '/kaggle/working/data'
CKPT_ROOT_NB4 = '/kaggle/working/checkpoints/regun_nb4'
os.makedirs(CKPT_ROOT_NB4, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

# Epoch cap: total compute <= 50 epochs equivalent
MAX_EPOCHS = 1 if TEST_MODE else 50

print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  Split seeds: {SPLIT_SEEDS}')

## B. Load Dataset, Splits & Helpers

In [ ]:
from utils import get_dataset, get_model, test, load_encoder_ckpt_safely, SubSet
from unlearn import unlear_func
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw):
    return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

UNLEARN_BS = 8 if TEST_MODE else 128

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=42, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=1.0, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=True, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='regun_cmf',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    def _stratified_subset(ds, fraction, seed=42):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_class = collections.defaultdict(list)
        for idx, lbl in enumerate(labels):
            by_class[int(lbl)].append(idx)
        kept = []
        for cls_idx in sorted(by_class):
            cls_pool = by_class[cls_idx]
            rng.shuffle(cls_pool)
            n_keep = max(1, math.ceil(len(cls_pool) * fraction))
            kept.extend(cls_pool[:n_keep])
        sub = torch.utils.data.Subset(ds, kept)
        base_targets = ds.targets if hasattr(ds, 'targets') else [
            ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_targets[i] for i in kept]
        return sub
    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: {TEST_FRACTION*100:.1f}% → Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=UNLEARN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, len(dataset_test)), num_workers=2, pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(
    dataset_train, batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
test_loader  = torch.utils.data.DataLoader(dataset_test, **TEST_KW)

# ── Load split files ─────────────────────────────────────────────────
splits = {}
for seed in SPLIT_SEEDS:
    split_file = f'{SPLIT_DIR}/forget_indices_seed{seed}.json'
    if not os.path.exists(split_file):
        raise FileNotFoundError(f'Split file missing: {split_file}')
    with open(split_file) as f:
        splits[seed] = json.load(f)
    print(f'Seed {seed}: forget={splits[seed]["n_forget"]}  retain={splits[seed]["n_retain"]}')

# ── Load Θ_o ─────────────────────────────────────────────────────────
args_pt = make_args(unlearn_method='pre_train', remove_FC=True, CMFClassifier=True)
orig_model = get_model(args_pt, device)
orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
orig_model.eval()
if hasattr(orig_model, 'recompute_cmf'):
    orig_model.recompute_cmf(train_loader, device=device)
print('\n── Original model accuracy ──')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES, set_name='Test')

## B-helper. Index-Based Eval Harness

Since the forget set spans ALL classes (cross-class split), accuracy is computed
over SAMPLE INDEX SETS, not class labels.

In [ ]:
def eval_on_indices(model, dataset, indices, device, batch_size=256):
    """Accuracy over an arbitrary set of sample indices (cross-class eval harness)."""
    if len(indices) == 0:
        return 0.0, 0, 0
    subset = SubSet(dataset, indices)
    loader = torch.utils.data.DataLoader(
        subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out  = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return correct / max(1, total), correct, total


def run_linear_probe_audit(model, retain_indices, dataset_train, dataset_test,
                            device, num_classes, args):
    """
    Train a FRESH linear probe on frozen encoder features (retain train set),
    evaluate on all test samples. Returns probe_retain_acc, probe_forget_acc.
    This is the audit probe used in cmf_dynamic's per-round logging.
    Reuses existing probe-eval code.
    """
    try:
        import evaluation
        from utils import get_model as _gm
        retain_ds = SubSet(dataset_train, retain_indices)
        retain_ldr = torch.utils.data.DataLoader(
            retain_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
        out = evaluation.run_linear_probe_on_fresh_clone(
            args=args,
            get_model_fn=_gm,
            device_probe=device,
            src_model=model,
            train_loader=retain_ldr,
            test_loader=torch.utils.data.DataLoader(
                dataset_test, batch_size=256, shuffle=False,
                num_workers=2, pin_memory=True),
            num_classes=num_classes,
            bs_probe=256,
        )
        return out.get('acc_test_retain', float('nan')), out.get('acc_test_forget', float('nan'))
    except Exception as e:
        print(f'  [probe audit failed: {e}]')
        return float('nan'), float('nan')


def cos_sim_avg(model, dataset, indices, device, batch_size=256):
    """
    Average cosine similarity between sample features and their assigned CMF class weight.
    Returns a float (or nan if CMF weights not available).
    """
    if not hasattr(model, 'CMFweights') or len(indices) == 0:
        return float('nan')
    subset = SubSet(dataset, indices)
    loader = torch.utils.data.DataLoader(
        subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    model.eval()
    sims = []
    W = F.normalize(model.CMFweights.weight, dim=1)  # [K, D]
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            f = model.extract_features(x)
            z = model._preprocess_feats_for_cmf(f)
            z = F.normalize(z, dim=1)
            # cosine similarity with assigned class weight
            sim = (z * W[y]).sum(dim=1)  # [B]
            sims.append(sim.cpu())
    return float(torch.cat(sims).mean().item())


print('Eval harness defined.')

## C. CMF Algorithm Implementations

### Paper context: what does the CMF formula use?

The paper (eq. 3, eq. 7) defines:
- μ_c = mean of φ_θ(x) over **all** training samples of class c (D = D_r ∪ D_f)
- W^CMF = [μ_1 ··· μ_K]^⊤

In the paper's setup (whole-class forgetting), the forget class is still present in the
training set, so its samples contribute to their own class mean — and the full training
loader naturally includes all classes. `mean_source='train'` reproduces this exactly.
`mean_source='retain'` is an ablation variant (removes forget samples from CMF means).

### Algorithm A — `cmf_static` (original paper, faithful)

Every epoch: rebuild W via closed-form CMF reconstruction from `mean_source` loader,
then **freeze W** for that epoch. Only θ (encoder) is updated.

> **Note:** The existing `recompute_cmf` in [`unlearn/cmf_weights.py`](unlearn/cmf_weights.py)
> L2-normalizes features BEFORE computing per-class means, then normalizes the centered
> means again. This is a **known deviation** from paper eq. 7 (which averages raw features).
> Figure 6 caption acknowledges this as 'an additional normalization step'. Preserved unchanged.

### Algorithm B — `cmf_dynamic` (new: AlternatingCMF)

W warm-started via CMF reconstruction, then becomes a real trainable parameter.
**At the start of every Phase 1**, W is reset to the paper's closed-form CMF solution
from `mean_source` loader — so Phase 1 always starts from the analytical baseline.
Phase 2 then applies gradient updates on top of that reset.

```
# warm-start W (pre-loop, uses mean_source loader)
model.recompute_cmf(mu_loader)
for r in range(rounds):
    # NEW: reset W to CMF analytical solution at start of each round
    model.recompute_cmf(mu_loader)          # always from mu_loader
    # Phase 1: freeze W (just reset), update θ for t_theta steps
    # Phase 2: freeze θ, gradient-update W on retain set for t_w steps
    # Audit: log probe retain/forget accuracy
```

This makes Phase 1 paper-faithful each round (W = CMF solution), while Phase 2
explores gradient refinement from that baseline. Total passes = rounds × (t_theta + t_w) ≤ 50.

In [ ]:
def cmf_static_unlearn(
    model, device,
    retain_loader, forget_loader,
    train_loader,
    args,
    epochs,
    base_method,          # 'scrub' | 'neggrad_plus' | 'random_label' | 'salun'
    mean_source,          # 'train' | 'retain'
    forget_indices,
    retain_indices,
    dataset_train,
    dataset_test,
):
    """
    Algorithm A — cmf_static: original paper's CMF method, reused unchanged.

    Every epoch:
      1. Rebuild W via closed-form CMF reconstruction from mean_source loader.
         mean_source='train'  → use train_loader (legacy, includes forget samples)
         mean_source='retain' → use retain_loader (fixed, forget-free)
      2. FREEZE W for this epoch.
      3. Update θ (encoder) via the base method's unlearning loss.

    NOTE: The L2-normalize-before-averaging step in recompute_cmf is preserved
    unchanged (known deviation from paper's Algorithm 1 raw-feature averaging).
    """
    assert hasattr(model, 'recompute_cmf'), 'cmf_static requires a ModelModule with recompute_cmf'

    mu_loader = train_loader if mean_source == 'train' else retain_loader

    # Optimizer for encoder only — W is a buffer, not a parameter
    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.lr, momentum=0.9, weight_decay=5e-4, nesterov=True
    )

    probe_retain_final, probe_forget_final = float('nan'), float('nan')
    epoch_logs = []

    forget_iterator = iter(forget_loader) if forget_loader is not None else None

    for epoch in range(1, epochs + 1):
        # ── Step 1: rebuild W (closed-form CMF), then freeze ─────────
        model.eval()
        model.recompute_cmf(mu_loader, device=device)
        # W is a buffer → always 'frozen' from gradient perspective;
        # no extra freeze needed — it has no requires_grad
        model.train()

        # ── Step 2: one epoch of base method loss on encoder ──────────
        total_loss, total_n = 0.0, 0
        for x, y in retain_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            # Use fixed (just-rebuilt) W — detach to be explicit
            W_fixed = model.CMFweights.weight.detach()

            # Retain loss (descent)
            f_r = model.extract_features(x)
            z_r = model._preprocess_feats_for_cmf(f_r)
            logits_r = (z_r @ W_fixed.t()) * getattr(model.args, 'temperature', 1.0)
            loss = F.cross_entropy(logits_r, y)

            # Forget loss (ascent) — if base method has ascent component
            if forget_loader is not None and base_method in ('neggrad_plus', 'scrub',
                                                              'random_label', 'salun',
                                                              'grad_ascent_descent'):
                try:
                    x_f, y_f = next(forget_iterator)
                except StopIteration:
                    forget_iterator = iter(forget_loader)
                    x_f, y_f = next(forget_iterator)
                x_f, y_f = x_f.to(device), y_f.to(device)

                if base_method == 'random_label':
                    # Relabel forget samples to random non-true labels
                    y_rand = torch.randint(0, NUM_CLASSES, y_f.shape, device=device)
                    same = (y_rand == y_f)
                    y_rand[same] = (y_rand[same] + 1) % NUM_CLASSES
                    f_f = model.extract_features(x_f)
                    z_f = model._preprocess_feats_for_cmf(f_f)
                    logits_f = (z_f @ W_fixed.t()) * getattr(model.args, 'temperature', 1.0)
                    loss_f = F.cross_entropy(logits_f, y_rand)
                    loss = loss + loss_f
                else:
                    # Gradient ascent on forget loss
                    f_f = model.extract_features(x_f)
                    z_f = model._preprocess_feats_for_cmf(f_f)
                    logits_f = (z_f @ W_fixed.t()) * getattr(model.args, 'temperature', 1.0)
                    loss_f = F.cross_entropy(logits_f, y_f)
                    loss = loss - loss_f

            loss.backward()
            if getattr(args, 'grad_norm_clip', None):
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_norm_clip)
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            total_n    += x.size(0)

        # ── Epoch eval ────────────────────────────────────────────────
        model.eval()
        model.recompute_cmf(mu_loader, device=device)
        ra, _, _ = eval_on_indices(model, dataset_train, retain_indices, device)
        fa, _, _ = eval_on_indices(model, dataset_train, forget_indices,  device)
        print(f'  [cmf_static epoch {epoch}/{epochs}] retain={ra:.4f}  forget={fa:.4f}  '
              f'loss={total_loss/max(1,total_n):.4f}')
        epoch_logs.append({'epoch': epoch, 'retain_acc': ra, 'forget_acc': fa})
        model.train()

    # Final probe audit
    model.eval()
    model.recompute_cmf(mu_loader, device=device)
    if not TEST_MODE:
        probe_retain_final, probe_forget_final = run_linear_probe_audit(
            model, retain_indices, dataset_train, dataset_test, device, NUM_CLASSES, args)

    return model, epoch_logs, probe_retain_final, probe_forget_final


print('cmf_static_unlearn defined.')

In [ ]:
def cmf_dynamic_unlearn(
    model, device,
    retain_loader, forget_loader,
    train_loader,
    args,
    base_method,
    mean_source,
    forget_indices,
    retain_indices,
    dataset_train,
    dataset_test,
    # cmf_dynamic config knobs
    rounds=5,
    t_theta=5,
    t_w=5,
    phase2_data='retain_only',   # 'retain_only' | 'retain_plus_forget'
    w_init_mode='cmf',           # 'cmf' | 'random'
):
    """
    Algorithm B — cmf_dynamic (AlternatingCMF).

    W is warm-started via one CMF reconstruction pass before the loop, then becomes a REAL
    trainable parameter updated via gradient descent in Phase 2.

    PAPER-FAITHFUL PER-ROUND RESET:
      At the START of Phase 1 in every round, W is rebuilt from scratch via closed-form
      CMF reconstruction (recompute_cmf) from mu_loader. This means Phase 1 always sees
      the analytical CMF solution, not the gradient-modified W from the previous round.
      Phase 2 then refines W via gradient steps from that CMF baseline.

    mean_source controls mu_loader for BOTH the pre-loop warm-start and per-round reset:
      'train'  → uses full train_loader (paper-faithful: eq.3 uses all samples D=D_r∪D_f)
      'retain' → uses retain_loader only (ablation: excludes forget samples from means)

    Alternating loop (with per-round reset):
      for r in range(rounds):
        RESET: recompute_cmf(mu_loader)  ← paper's analytical W
        Phase 1: W frozen (reset value), update θ for t_theta steps via base method loss
        Phase 2: freeze θ, update W via CE on retain set for t_w steps
        Audit:   train fresh probe on frozen θ → log probe retain/forget acc

    Compute budget: rounds * (t_theta + t_w) <= MAX_EPOCHS

    NOTE: L2-normalize-before-averaging in recompute_cmf is preserved unchanged
    (known deviation from paper eq.7 raw-feature averaging; see Figure 6 caption).
    """
    assert hasattr(model, 'CMFweights'), 'cmf_dynamic requires ModelModule with CMFweights'

    # ── Compute budget audit ──────────────────────────────────────────
    total_epoch_equiv = rounds * (t_theta + t_w)
    assert total_epoch_equiv <= MAX_EPOCHS, (
        f'cmf_dynamic budget violation: rounds({rounds}) × (t_theta({t_theta}) + t_w({t_w})) '
        f'= {total_epoch_equiv} > MAX_EPOCHS={MAX_EPOCHS}')
    print(f'  [cmf_dynamic] budget: {rounds} rounds × ({t_theta}+{t_w}) = '
          f'{total_epoch_equiv} epoch-equiv (≤ {MAX_EPOCHS} cap)  ✓')

    # mu_loader controls CMF reconstruction for BOTH warm-start and per-round reset.
    # 'train' = paper-faithful (eq.3 uses full D=D_r∪D_f); 'retain' = ablation variant.
    mu_loader = train_loader if mean_source == 'train' else retain_loader

    # ── Step 0: pre-loop warm-start W ────────────────────────────────
    # This initialises W before the alternating loop begins.
    # w_init_mode isolates whether this initial seed matters at all,
    # since every round resets W to the CMF solution anyway.
    model.eval()
    if w_init_mode == 'cmf':
        # Closed-form CMF reconstruction (same formula as cmf_static)
        model.recompute_cmf(mu_loader, device=device)
        print(f'  [cmf_dynamic] W pre-loop warm-start: CMF reconstruction '
              f'(mean_source={mean_source})')
    else:
        # Random init: standard normal, then L2-normalize
        with torch.no_grad():
            torch.nn.init.normal_(model.CMFweights.weight, 0, 0.01)
            model.CMFweights.weight.copy_(
                F.normalize(model.CMFweights.weight, dim=1))
        print(f'  [cmf_dynamic] W pre-loop warm-start: RANDOM init (w_init_mode=random)')
    print(f'  [cmf_dynamic] NOTE: per-round CMF reset always uses mu_loader '
          f'(mean_source={mean_source}) regardless of w_init_mode')

    # ── Make W a real nn.Parameter ────────────────────────────────────
    # CMFWeights stores W as a buffer; we wrap it as a parameter for Phase 2.
    W_data = model.CMFweights.weight.data.clone().requires_grad_(False)
    W_param = nn.Parameter(W_data.clone())
    # Remove old buffer and register as parameter
    del model.CMFweights._buffers['weight']
    model.CMFweights.register_parameter('weight', W_param)

    # ── Optimizers ───────────────────────────────────────────────────
    encoder_params = [p for n, p in model.named_parameters()
                      if 'CMFweights' not in n and p.requires_grad]
    w_params       = [model.CMFweights.weight]

    opt_theta = optim.SGD(encoder_params, lr=args.lr,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)
    opt_w     = optim.SGD(w_params, lr=args.lr * 0.1,
                          momentum=0.9, weight_decay=0.0)

    round_logs = []
    forget_iterator = iter(forget_loader) if forget_loader is not None else None

    for r in range(1, rounds + 1):
        print(f'\n  [cmf_dynamic] ── Round {r}/{rounds} ──────────────────────')

        # ── Per-round CMF reset: rebuild W from analytical solution ──
        # Paper-faithful: W = CMF(mu_loader) at the start of every Phase 1.
        # This replaces any gradient-modified W from the previous round's Phase 2,
        # so Phase 1 always starts from the paper's closed-form CMF baseline.
        # NOTE: uses mu_loader (controlled by mean_source), NOT hardcoded to retain.
        model.eval()
        # recompute_cmf writes directly into model.CMFweights.weight via .copy_(),
        # which works whether weight is a buffer or an nn.Parameter (both support .copy_()).
        model.recompute_cmf(mu_loader, device=device)

        # ── Phase 1: update θ, W frozen (CMF-reset value) ────────────
        model.CMFweights.weight.requires_grad_(False)
        model.train()
        for t in range(t_theta):
            for x, y in retain_loader:
                x, y = x.to(device), y.to(device)
                opt_theta.zero_grad()

                W_fixed = model.CMFweights.weight.detach()
                f_r = model.extract_features(x)
                z_r = model._preprocess_feats_for_cmf(f_r)
                logits_r = (z_r @ W_fixed.t()) * getattr(model.args, 'temperature', 1.0)
                loss = F.cross_entropy(logits_r, y)

                # Forget loss depending on base_method
                if forget_loader is not None and base_method in (
                        'neggrad_plus', 'scrub', 'random_label', 'salun',
                        'grad_ascent_descent'):
                    try:
                        x_f, y_f = next(forget_iterator)
                    except StopIteration:
                        forget_iterator = iter(forget_loader)
                        x_f, y_f = next(forget_iterator)
                    x_f, y_f = x_f.to(device), y_f.to(device)
                    f_f = model.extract_features(x_f)
                    z_f = model._preprocess_feats_for_cmf(f_f)
                    logits_f = (z_f @ W_fixed.t()) * getattr(model.args, 'temperature', 1.0)
                    if base_method == 'random_label':
                        y_rand = torch.randint(0, NUM_CLASSES, y_f.shape, device=device)
                        same = (y_rand == y_f)
                        y_rand[same] = (y_rand[same] + 1) % NUM_CLASSES
                        loss = loss + F.cross_entropy(logits_f, y_rand)
                    else:
                        loss = loss - F.cross_entropy(logits_f, y_f)

                loss.backward()
                if getattr(args, 'grad_norm_clip', None):
                    torch.nn.utils.clip_grad_norm_(encoder_params, args.grad_norm_clip)
                opt_theta.step()
                break  # one batch per step for efficiency within round

        # ── Phase 2: update W via gradient, θ frozen ──────────────────
        # This is REAL gradient updates on W — not closed-form reconstruction.
        for p in encoder_params:
            p.requires_grad_(False)
        model.CMFweights.weight.requires_grad_(True)
        model.train()

        p2_loaders = [retain_loader]
        if phase2_data == 'retain_plus_forget' and forget_loader is not None:
            p2_loaders.append(forget_loader)

        p2_iter_list = [iter(ldr) for ldr in p2_loaders]
        for t in range(t_w):
            for p2_iter in p2_iter_list:
                try:
                    x_p2, y_p2 = next(p2_iter)
                except StopIteration:
                    p2_iter = iter(p2_loaders[p2_iter_list.index(p2_iter)])
                    x_p2, y_p2 = next(p2_iter)
                x_p2, y_p2 = x_p2.to(device), y_p2.to(device)
                opt_w.zero_grad()
                with torch.no_grad():
                    f_p2 = model.extract_features(x_p2)
                    z_p2 = model._preprocess_feats_for_cmf(f_p2)
                W = model.CMFweights.weight
                logits_p2 = (z_p2 @ W.t()) * getattr(model.args, 'temperature', 1.0)
                loss_w = F.cross_entropy(logits_p2, y_p2)
                loss_w.backward()
                opt_w.step()
                # Re-normalize W after update
                with torch.no_grad():
                    model.CMFweights.weight.copy_(
                        F.normalize(model.CMFweights.weight.data, dim=1))
                break  # one batch per step

        # Unfreeze encoder for next round's Phase 1
        for p in encoder_params:
            p.requires_grad_(True)
        model.CMFweights.weight.requires_grad_(False)

        # ── Audit (logging only): train fresh probe on frozen θ ───────
        model.eval()
        ra, _, _ = eval_on_indices(model, dataset_train, retain_indices, device)
        fa, _, _ = eval_on_indices(model, dataset_train, forget_indices,  device)
        print(f'  [round {r}] output: retain={ra:.4f}  forget={fa:.4f}')

        probe_r, probe_f = float('nan'), float('nan')
        if not TEST_MODE:
            probe_r, probe_f = run_linear_probe_audit(
                model, retain_indices, dataset_train, dataset_test,
                device, NUM_CLASSES, args)
            print(f'  [round {r}] probe : retain={probe_r:.4f}  forget={probe_f:.4f}')

        round_logs.append({
            'round': r,
            'output_retain_acc': ra, 'output_forget_acc': fa,
            'probe_retain_acc': probe_r, 'probe_forget_acc': probe_f,
        })
        model.train()

    # Final eval
    model.eval()
    probe_retain_final = round_logs[-1]['probe_retain_acc'] if round_logs else float('nan')
    probe_forget_final = round_logs[-1]['probe_forget_acc'] if round_logs else float('nan')

    return model, round_logs, probe_retain_final, probe_forget_final


print('cmf_dynamic_unlearn defined.')

## D. Experiment Matrix

```
for base_method in [scrub, neggrad_plus, random_label, salun]:
  for classifier_strategy in [cmf_static, cmf_dynamic]:
    for mean_source in [train, retain]:
      for seed in [0, 1, 2]:
        run on fixed 3:7 split
```

Priority: scrub full sub-matrix first, then random_label, then neggrad_plus/salun.

In [ ]:
# ── Learning rates for CMF methods ───────────────────────────────────
_CMF_LR = {
    'scrub':        {'cifar10': 5e-3, 'cifar100': 5e-3, 'tinyimagenet': 5e-3},
    'neggrad_plus': {'cifar10': 1e-4, 'cifar100': 1e-4, 'tinyimagenet': 3e-5},
    'random_label': {'cifar10': 1e-4, 'cifar100': 2e-3, 'tinyimagenet': 1e-2},
    'salun':        {'cifar10': 2e-4, 'cifar100': 2e-3, 'tinyimagenet': 1e-2},
}

# ── Epoch budget for cmf_static ───────────────────────────────────────
_STATIC_EPOCHS = {
    'scrub': 3, 'neggrad_plus': 3, 'random_label': 4, 'salun': 4,
}
if TEST_MODE:
    _STATIC_EPOCHS = {k: 1 for k in _STATIC_EPOCHS}

# ── cmf_dynamic round config (rounds × (t_theta+t_w) ≤ MAX_EPOCHS) ───
if TEST_MODE:
    _DYN_CFG = {'rounds': 1, 't_theta': 1, 't_w': 1}  # 1×(1+1)=2 ≤ 50
else:
    _DYN_CFG = {'rounds': 5, 't_theta': 5, 't_w': 5}  # 5×(5+5)=50 ≤ 50

print(f'cmf_static epochs: {_STATIC_EPOCHS}')
print(f'cmf_dynamic config: {_DYN_CFG}')
print(f'cmf_dynamic total epoch-equiv: '
      f'{_DYN_CFG["rounds"]}×({_DYN_CFG["t_theta"]}+{_DYN_CFG["t_w"]}) = '
      f'{_DYN_CFG["rounds"]*(_DYN_CFG["t_theta"]+_DYN_CFG["t_w"])} ≤ {MAX_EPOCHS}  ✓')

# ── Experiment matrix ─────────────────────────────────────────────────
# Prioritize scrub first, then random_label, then neggrad_plus/salun
BASE_METHODS         = ['scrub', 'random_label', 'neggrad_plus', 'salun']
CLASSIFIER_STRATEGIES = ['cmf_static', 'cmf_dynamic']
MEAN_SOURCES         = ['train', 'retain']

# cmf_dynamic extra ablation knobs
# w_init_mode controls the PRE-LOOP warm-start only.
# The per-round reset always uses recompute_cmf(mu_loader), so comparing
# w_init_mode='cmf' vs 'random' isolates whether the single pre-loop seed matters
# (it should not, since round 1 immediately resets W to CMF anyway).
W_INIT_MODES    = ['cmf', 'random']   # isolate whether pre-loop warm-start matters
PHASE2_DATA     = 'retain_only'       # 'retain_only' | 'retain_plus_forget'

total_runs = (len(BASE_METHODS) * len(CLASSIFIER_STRATEGIES) *
              len(MEAN_SOURCES) * len(SPLIT_SEEDS))
print(f'\nTotal experiment matrix: {total_runs} runs '
      f'(+ {len(BASE_METHODS)*len(MEAN_SOURCES)*len(SPLIT_SEEDS)} extra w_init_mode=random runs)')

In [ ]:
ALL_RESULTS = []

for base_method in BASE_METHODS:
    # Map internal method name to unlear_func key
    _method_key_map = {
        'scrub': 'scrub',
        'neggrad_plus': 'grad_ascent_descent',
        'random_label': 'random_label',
        'salun': 'salun',
    }
    lr = _CMF_LR[base_method][DATASET]

    for classifier_strategy in CLASSIFIER_STRATEGIES:
        # For cmf_dynamic, run both w_init_modes; for cmf_static, just one
        w_init_modes_to_run = (W_INIT_MODES if classifier_strategy == 'cmf_dynamic'
                               else ['cmf'])

        for mean_source in MEAN_SOURCES:
            for seed in SPLIT_SEEDS:
                for w_init_mode in w_init_modes_to_run:
                    # Skip w_init_mode for cmf_static (always 'cmf')
                    if classifier_strategy == 'cmf_static' and w_init_mode != 'cmf':
                        continue

                    run_id = (f'{base_method}__{classifier_strategy}__'
                              f'{mean_source}__seed{seed}'
                              + (f'__winit_{w_init_mode}'
                                 if classifier_strategy == 'cmf_dynamic' else ''))

                    print(f'\n{"#"*65}')
                    print(f'  {run_id}')
                    print(f'{"#"*65}')

                    split = splits[seed]
                    retain_indices = split['retain_indices']
                    forget_indices = split['forget_indices']

                    retain_ds = SubSet(dataset_train, retain_indices)
                    forget_ds = SubSet(dataset_train, forget_indices)
                    retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
                    forget_loader = torch.utils.data.DataLoader(forget_ds, **LOADER_KW)

                    args_run = make_args(
                        unlearn_method=f'{classifier_strategy}_{base_method}',
                        epochs_or_steps=_STATIC_EPOCHS.get(base_method, 3),
                        lr=lr,
                        batch_size=UNLEARN_BS,
                        num_retain_samples=len(retain_indices),
                        num_forget_samples=len(forget_indices),
                        unlearn_class=[],
                        remove_FC=True, CMFClassifier=True,
                        grad_norm_clip=1.0,
                    )

                    # Fresh copy of Θ_o
                    model = get_model(args_run, device)
                    model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))

                    t0 = time.time()
                    try:
                        if classifier_strategy == 'cmf_static':
                            epochs = min(_STATIC_EPOCHS.get(base_method, 3), MAX_EPOCHS)
                            unlearnt, epoch_logs, probe_r, probe_f = cmf_static_unlearn(
                                model=model, device=device,
                                retain_loader=retain_loader,
                                forget_loader=forget_loader,
                                train_loader=train_loader,
                                args=args_run,
                                epochs=epochs,
                                base_method=_method_key_map.get(base_method, base_method),
                                mean_source=mean_source,
                                forget_indices=forget_indices,
                                retain_indices=retain_indices,
                                dataset_train=dataset_train,
                                dataset_test=dataset_test,
                            )
                        else:  # cmf_dynamic
                            unlearnt, epoch_logs, probe_r, probe_f = cmf_dynamic_unlearn(
                                model=model, device=device,
                                retain_loader=retain_loader,
                                forget_loader=forget_loader,
                                train_loader=train_loader,
                                args=args_run,
                                base_method=_method_key_map.get(base_method, base_method),
                                mean_source=mean_source,
                                forget_indices=forget_indices,
                                retain_indices=retain_indices,
                                dataset_train=dataset_train,
                                dataset_test=dataset_test,
                                rounds=_DYN_CFG['rounds'],
                                t_theta=_DYN_CFG['t_theta'],
                                t_w=_DYN_CFG['t_w'],
                                phase2_data=PHASE2_DATA,
                                w_init_mode=w_init_mode,
                            )
                    except Exception as e:
                        import traceback
                        print(f'  ERROR in {run_id}: {e}')
                        traceback.print_exc()
                        ALL_RESULTS.append(dict(
                            base_method=base_method,
                            classifier_strategy=classifier_strategy,
                            mean_source=mean_source,
                            seed=seed,
                            w_init_mode=w_init_mode,
                            output_retain_acc=float('nan'),
                            output_forget_acc=float('nan'),
                            probe_retain_acc=float('nan'),
                            probe_forget_acc=float('nan'),
                            ncc_retain_acc=float('nan'),
                            ncc_forget_acc=float('nan'),
                            retain_cos_sim_avg=float('nan'),
                            forget_cos_sim_avg=float('nan'),
                            wall_clock_minutes=float('nan'),
                            error=str(e),
                        ))
                        continue

                    elapsed = time.time() - t0

                    # ── Save checkpoint ───────────────────────────────
                    ckpt_dir = f'{CKPT_ROOT_NB4}/{base_method}/{classifier_strategy}'
                    os.makedirs(ckpt_dir, exist_ok=True)
                    ckpt_out = (f'{ckpt_dir}/{DATASET}_{ARCH}_{_MODE_TAG}_'
                                f'{mean_source}_seed{seed}'
                                + (f'_winit_{w_init_mode}'
                                   if classifier_strategy == 'cmf_dynamic' else '')
                                + '.pt')
                    torch.save(unlearnt.state_dict(), ckpt_out)

                    # ── Output accuracy over index sets ───────────────
                    unlearnt.eval()
                    out_ra, _, _ = eval_on_indices(unlearnt, dataset_train, retain_indices, device)
                    out_fa, _, _ = eval_on_indices(unlearnt, dataset_train, forget_indices,  device)

                    # ── Cosine similarity ─────────────────────────────
                    cos_r = cos_sim_avg(unlearnt, dataset_train, retain_indices, device)
                    cos_f = cos_sim_avg(unlearnt, dataset_train, forget_indices,  device)

                    # ── Log budget config in every run ────────────────
                    budget_log = {
                        'MAX_EPOCHS': MAX_EPOCHS,
                        'strategy': classifier_strategy,
                    }
                    if classifier_strategy == 'cmf_static':
                        budget_log['epochs'] = _STATIC_EPOCHS.get(base_method, 3)
                        budget_log['epoch_equiv'] = budget_log['epochs']
                    else:
                        budget_log.update({
                            'rounds': _DYN_CFG['rounds'],
                            't_theta': _DYN_CFG['t_theta'],
                            't_w': _DYN_CFG['t_w'],
                            'epoch_equiv': _DYN_CFG['rounds'] * (
                                _DYN_CFG['t_theta'] + _DYN_CFG['t_w']),
                            'w_init_mode': w_init_mode,
                            'phase2_data': PHASE2_DATA,
                            'per_round_cmf_reset': True,
                            'per_round_reset_source': mean_source,
                        })
                    print(f'  budget_log: {budget_log}')

                    print(f'  → output: retain={out_ra:.4f}  forget={out_fa:.4f}  '
                          f'probe: retain={probe_r:.4f}  forget={probe_f:.4f}  '
                          f'({elapsed/60:.1f} min)')

                    ALL_RESULTS.append(dict(
                        base_method=base_method,
                        classifier_strategy=classifier_strategy,
                        mean_source=mean_source,
                        seed=seed,
                        w_init_mode=w_init_mode,
                        output_retain_acc=out_ra,
                        output_forget_acc=out_fa,
                        probe_retain_acc=probe_r,
                        probe_forget_acc=probe_f,
                        ncc_retain_acc=float('nan'),   # NCC disabled for speed
                        ncc_forget_acc=float('nan'),
                        retain_cos_sim_avg=cos_r,
                        forget_cos_sim_avg=cos_f,
                        wall_clock_minutes=elapsed/60,
                        budget_epoch_equiv=budget_log.get('epoch_equiv', 0),
                    ))

print(f'\nAll runs done. {len(ALL_RESULTS)} results collected.')

## E. Results: CSV + Pivoted cmf_static vs cmf_dynamic Comparison Table

In [ ]:
results_df = pd.DataFrame(ALL_RESULTS)
csv_path = f'/kaggle/working/results_nb4_{DATASET}_{ARCH}.csv'
results_df.to_csv(csv_path, index=False)
print(f'Full results saved: {csv_path}')
print(f'Shape: {results_df.shape}')
print(results_df[[
    'base_method', 'classifier_strategy', 'mean_source', 'seed',
    'output_retain_acc', 'output_forget_acc',
    'probe_retain_acc', 'probe_forget_acc'
]].to_string(index=False))

In [ ]:
# ── Pivoted table: cmf_static vs cmf_dynamic side by side ─────────────
# Filter to w_init_mode='cmf' for a clean apples-to-apples comparison
df_main = results_df[results_df['w_init_mode'] == 'cmf'].copy()

_metric_cols = [c for c in [
    'output_retain_acc', 'output_forget_acc',
    'probe_retain_acc', 'probe_forget_acc',
    'retain_cos_sim_avg', 'forget_cos_sim_avg',
] if c in df_main.columns and df_main[c].notna().any()]

# Mean ± std over seeds
agg = df_main.groupby(
    ['base_method', 'classifier_strategy', 'mean_source']
)[_metric_cols].agg(['mean', 'std'])

# Flatten multi-index columns
agg.columns = ['_'.join(c) for c in agg.columns]
agg = agg.reset_index()

# Pivot so cmf_static and cmf_dynamic sit side by side
pivot_rows = []
for base_method in BASE_METHODS:
    for mean_src in MEAN_SOURCES:
        row = {'base_method': base_method, 'mean_source': mean_src}
        for strat in ['cmf_static', 'cmf_dynamic']:
            sub = agg[
                (agg['base_method'] == base_method) &
                (agg['classifier_strategy'] == strat) &
                (agg['mean_source'] == mean_src)
            ]
            if len(sub) == 0:
                continue
            s = sub.iloc[0]
            for col in ['output_retain_acc', 'output_forget_acc',
                        'probe_retain_acc', 'probe_forget_acc']:
                m_col = f'{col}_mean'
                s_col = f'{col}_std'
                if m_col in s.index:
                    m_val = s.get(m_col, float('nan'))
                    s_val = s.get(s_col, float('nan'))
                    row[f'{strat}_{col}'] = (
                        f"{m_val:.3f}±{s_val:.3f}"
                        if not (pd.isna(m_val) or pd.isna(s_val)) else 'N/A'
                    )
        pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows)

# Pretty display
print('\n' + '='*80)
print(f'CMF STRATEGY COMPARISON — {DATASET}/{ARCH}')
print(f'(mean±std over {len(SPLIT_SEEDS)} seeds, w_init_mode=cmf only)')
print('='*80)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
print(pivot_df.to_string(index=False))

# Save pivot table
pivot_path = f'/kaggle/working/pivot_cmf_comparison_{DATASET}_{ARCH}.csv'
pivot_df.to_csv(pivot_path, index=False)
print(f'\nPivot table saved: {pivot_path}')

In [ ]:
# ── Additional table: cmf_dynamic w_init_mode='cmf' vs 'random' ───────
df_dyn = results_df[results_df['classifier_strategy'] == 'cmf_dynamic'].copy()
if len(df_dyn) > 0:
    agg_w = df_dyn.groupby(
        ['base_method', 'mean_source', 'w_init_mode']
    )[['output_retain_acc', 'output_forget_acc',
       'probe_retain_acc', 'probe_forget_acc']].agg(['mean', 'std'])
    agg_w.columns = ['_'.join(c) for c in agg_w.columns]
    print('\n=== cmf_dynamic: CMF vs Random W initialisation ===')
    print(agg_w.round(4).to_string())

In [ ]:
# ── Bar chart: cmf_static vs cmf_dynamic per base_method ─────────────
df_plot = df_main.groupby(
    ['base_method', 'classifier_strategy']
)[['output_retain_acc', 'output_forget_acc']].mean().reset_index()

n_methods = len(BASE_METHODS)
fig, axes = plt.subplots(1, n_methods, figsize=(4*n_methods, 5), sharey=True)
if n_methods == 1:
    axes = [axes]

for ax, bm in zip(axes, BASE_METHODS):
    sub = df_plot[df_plot['base_method'] == bm]
    strategies = sub['classifier_strategy'].tolist()
    x = np.arange(len(strategies))
    w = 0.35
    ax.bar(x - w/2, sub['output_retain_acc'].values, w,
           label='Retain', color='steelblue', alpha=0.85)
    ax.bar(x + w/2, sub['output_forget_acc'].values, w,
           label='Forget', color='tomato', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(['Static', 'Dynamic'], fontsize=10)
    ax.set_title(bm, fontsize=11)
    ax.set_ylim(0, 1.1)
    if ax == axes[0]:
        ax.set_ylabel('Accuracy')
        ax.legend(fontsize=9)

fig.suptitle(f'cmf_static vs cmf_dynamic\n{DATASET}/{ARCH} (mean over {len(SPLIT_SEEDS)} seeds)',
             fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/chart_cmf_comparison.png', dpi=120)
plt.show()
print('Chart saved.')

## F. Written Summary: Questions (a)–(d)

In [ ]:
def _fmt(v):
    try: return f'{float(v):.4f}'
    except: return 'N/A'

def _get(df, base, strat, src, metric):
    sub = df[
        (df['base_method']==base) &
        (df['classifier_strategy']==strat) &
        (df['mean_source']==src) &
        (df['w_init_mode']=='cmf')
    ]
    if len(sub) == 0: return float('nan')
    return sub[metric].mean()

summary_lines = []
summary_lines.append('='*70)
summary_lines.append(f'  WRITTEN SUMMARY — ReGUn CMF Ablation')
summary_lines.append(f'  Dataset: {DATASET}  Arch: {ARCH}')
summary_lines.append(f'  Split: {FORGET_FRACTION*100:.0f}% forget / {(1-FORGET_FRACTION)*100:.0f}% retain '
                     f'(stratified random cross-class)')
summary_lines.append(f'  Seeds: {SPLIT_SEEDS}  (mean over {len(SPLIT_SEEDS)} seeds per config)')
summary_lines.append('='*70)

# (a) cmf_dynamic retain accuracy vs cmf_static
summary_lines.append('\n(a) Does cmf_dynamic achieve HIGHER RETAIN ACCURACY than cmf_static?')
summary_lines.append('    (same base_method, 3:7 mixed-class split, mean_source=retain)')
for bm in BASE_METHODS:
    ra_static  = _get(results_df, bm, 'cmf_static',  'retain', 'output_retain_acc')
    ra_dynamic = _get(results_df, bm, 'cmf_dynamic', 'retain', 'output_retain_acc')
    diff = ra_dynamic - ra_static if not (pd.isna(ra_static) or pd.isna(ra_dynamic)) else float('nan')
    direction = ('higher ↑' if diff > 0.005 else
                 'lower ↓'  if diff < -0.005 else
                 'comparable ≈' if not pd.isna(diff) else 'N/A')
    summary_lines.append(f'  {bm:15s}: static={_fmt(ra_static)}  '
                         f'dynamic={_fmt(ra_dynamic)}  '
                         f'Δ={_fmt(diff)}  → {direction}')

# (b) output-vs-probe forget gap ("illusion" pattern)
summary_lines.append('\n(b) Does cmf_dynamic re-open an output-vs-probe FORGET GAP?')
summary_lines.append('    ("illusion" pattern: output forget acc low, but probe forget acc high)')
for bm in BASE_METHODS:
    for strat in ['cmf_static', 'cmf_dynamic']:
        out_f  = _get(results_df, bm, strat, 'retain', 'output_forget_acc')
        prob_f = _get(results_df, bm, strat, 'retain', 'probe_forget_acc')
        gap = prob_f - out_f if not (pd.isna(prob_f) or pd.isna(out_f)) else float('nan')
        illusion = 'YES — gap detected ⚠' if (not pd.isna(gap)) and gap > 0.05 else \
                   'no clear gap ✓'       if (not pd.isna(gap)) else 'N/A (probe not run)'
        summary_lines.append(f'  {bm:15s}/{strat:13s}: '
                             f'output_forget={_fmt(out_f)}  '
                             f'probe_forget={_fmt(prob_f)}  '
                             f'gap={_fmt(gap)}  → {illusion}')

# (c) mean_source='retain' vs 'train'
summary_lines.append('\n(c) Does mean_source="retain" vs "train" visibly matter at 30% forget?')
for bm in BASE_METHODS:
    for strat in ['cmf_static', 'cmf_dynamic']:
        ra_train  = _get(results_df, bm, strat, 'train',  'output_retain_acc')
        ra_retain = _get(results_df, bm, strat, 'retain', 'output_retain_acc')
        fa_train  = _get(results_df, bm, strat, 'train',  'output_forget_acc')
        fa_retain = _get(results_df, bm, strat, 'retain', 'output_forget_acc')
        diff_r = ra_retain - ra_train if not (pd.isna(ra_retain) or pd.isna(ra_train)) else float('nan')
        diff_f = fa_retain - fa_train if not (pd.isna(fa_retain) or pd.isna(fa_train)) else float('nan')
        summary_lines.append(f'  {bm:15s}/{strat:13s}: '
                             f'retain_acc Δ(retain-train)={_fmt(diff_r)}  '
                             f'forget_acc Δ={_fmt(diff_f)}')

# (d) w_init_mode for cmf_dynamic
summary_lines.append('\n(d) Does w_init_mode ("cmf" vs "random") change cmf_dynamic outcome?')
df_dyn_all = results_df[results_df['classifier_strategy'] == 'cmf_dynamic']
for bm in BASE_METHODS:
    for src in MEAN_SOURCES:
        ra_cmf  = df_dyn_all[
            (df_dyn_all['base_method']==bm) &
            (df_dyn_all['mean_source']==src) &
            (df_dyn_all['w_init_mode']=='cmf')
        ]['output_retain_acc'].mean()
        ra_rand = df_dyn_all[
            (df_dyn_all['base_method']==bm) &
            (df_dyn_all['mean_source']==src) &
            (df_dyn_all['w_init_mode']=='random')
        ]['output_retain_acc'].mean()
        diff = ra_rand - ra_cmf if not (pd.isna(ra_rand) or pd.isna(ra_cmf)) else float('nan')
        matters = ('YES — warm-start matters ↑' if (not pd.isna(diff)) and abs(diff) > 0.01 else
                   'no significant difference ≈' if not pd.isna(diff) else 'N/A')
        summary_lines.append(f'  {bm:15s}/src={src:6s}: '
                             f'cmf_init={_fmt(ra_cmf)}  '
                             f'rand_init={_fmt(ra_rand)}  '
                             f'Δ={_fmt(diff)}  → {matters}')

summary_lines.append('\n' + '='*70)
SUMMARY_TEXT = '\n'.join(summary_lines)
print(SUMMARY_TEXT)

# Save summary
summary_path = f'/kaggle/working/cmf_summary_{DATASET}_{ARCH}.txt'
with open(summary_path, 'w') as f:
    f.write(SUMMARY_TEXT)
print(f'\nSummary saved: {summary_path}')